# Scheduling a Notebook as a Job in Databricks

1. Load the data
2. Transform the data
3. Persist the data
4. Delta transaction logs

### 1. Load the Data

**Create a Temp View and Query with**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_view AS
SELECT * FROM accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric;

SELECT * FROM temp_view LIMIT 5;

**Create a PySpark Dataframe**

In [0]:
df = spark.table("accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric")

display(df.limit(5))

**Read CSV into Dataframe**

In [0]:
df = spark.read.csv("/Volumes/databricks_day/weather/files/weather.csv", header=True, inferSchema=True)

display(df.limit(5))

**Read EXCEL into Dataframe**

In [0]:
df = spark.read.excel("/Volumes/databricks_day/weather/files/weather.xlsx")

display(df.limit(5))


### 2. Transform the Data

**Load Fresh Copy**

In [0]:
df = spark.table("accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric")

display(df.limit(5))

**Keep Only Select Columns**

In [0]:
df = df.select("latitude", "longitude", "station_code", "date", "phrase_long", "temperature_avg", "wind_speed_avg", "precipitation_probability", "rain_lwe_total")

display(df.limit(5))

**Drop rows with NULL and Station NYCG**

In [0]:
df = df.dropna().filter(df.station_code != "NYCG")

display(df.limit(5))

**Create Dataframe of Monthly Climate Averages**

In [0]:
from pyspark.sql.functions import month, avg

df_monthly_avg = df.withColumn("month", month("date")) \
    .groupBy("station_code", "month") \
    .agg(
        avg("temperature_avg").alias("avg_temperature"),
        avg("wind_speed_avg").alias("avg_wind_speed"),
        avg("precipitation_probability").alias("avg_precip_probability"),
        avg("rain_lwe_total").alias("avg_rain_lwe")
    )

display(df_monthly_avg.limit(5))

### 3. Persist the Data


**Create a View**

In [0]:
%sql
CREATE OR REPLACE VIEW databricks_day.weather.monthly_avg_view AS
SELECT
    station_code,
    MONTH(date) AS month,
    AVG(temperature_avg) AS avg_temperature,
    AVG(wind_speed_avg) AS avg_wind_speed,
    AVG(precipitation_probability) AS avg_precip_probability,
    AVG(rain_lwe_total) AS avg_rain_lwe
FROM accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric
GROUP BY station_code, MONTH(date);

SELECT * FROM databricks_day.weather.monthly_avg_view LIMIT 5;

**Create a Table**

In [0]:
df_monthly_avg.write.mode("overwrite").saveAsTable("databricks_day.weather.weather_monthly_avg")

df = spark.table("databricks_day.weather.weather_monthly_avg")

display(df.limit(5))

print("Number of rows in the table: ", df.count())

**What if we want to merge new data**

In [0]:
from pyspark.sql import Row
from delta.tables import DeltaTable

# Create new rows to add
new_data = [
    Row(station_code="ORNY", month=4, avg_temperature=14.174545, avg_wind_speed=2.803636, avg_precip_probability=17.90909090909091, avg_rain_lwe=0.078509091),
    Row(station_code="ABC", month=4, avg_temperature=15.132727, avg_wind_speed=2.845455, avg_precip_probability=41.81818181818182, avg_rain_lwe=2.496131818)
]

df_new = spark.createDataFrame(new_data)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "databricks_day.weather.weather_monthly_avg")

delta_table.alias("target").merge(
    df_new.alias("source"),
    "target.station_code = source.station_code AND target.month = source.month"
).whenMatchedUpdate(set={
    "avg_temperature": "source.avg_temperature",
    "avg_wind_speed": "source.avg_wind_speed",
    "avg_precip_probability": "source.avg_precip_probability",
    "avg_rain_lwe": "source.avg_rain_lwe"
}).whenNotMatchedInsert(values={
    "station_code": "source.station_code",
    "month": "source.month",
    "avg_temperature": "source.avg_temperature",
    "avg_wind_speed": "source.avg_wind_speed",
    "avg_precip_probability": "source.avg_precip_probability",
    "avg_rain_lwe": "source.avg_rain_lwe"
}).execute()

display(delta_table.history())